In [1]:

# ============================================================
# Shared setup (anonymized; paths are never printed)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

# Grayscale, publication style; no colour, no titles/captions in saved images.
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"image.cmap": "gray", "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 11, "savefig.dpi": 600})

DATA = Path("../data"); FIG = Path("../results/figures"); TAB = Path("../results/tables")
RNG = np.random.default_rng(42)

# Dataset registry (public sources) and decision domain.
DATASETS = {"telco": "telco_churn.csv", "adult": "adult.parquet",
            "credit_risk": "credit_risk.csv", "gmsc": "gmsc.parquet",
            "taiwan": "taiwan.parquet"}
DOMAIN = {"telco": "non-credit", "adult": "non-credit",
          "credit_risk": "consumer credit", "gmsc": "consumer credit",
          "taiwan": "consumer credit"}
# Grayscale shade + marker per domain (colour-free, print-safe).
STYLE = {"consumer credit": ("0.15", "o"), "corporate credit": ("0.0", "*"),
         "non-credit": ("0.45", "s")}
# Paper-1 anchor (Polish corporate bankruptcy; values reported in the first paper).
POLISH = {"gini": 2*0.778-1, "pbar": 0.0482, "naive_gap": 2.72, "indep_gap": 0.33}

def load(name):
    f = DATA / DATASETS[name]
    return pd.read_parquet(f) if str(f).endswith(".parquet") else pd.read_csv(f)

def prep(name, df):
    # Return numeric design matrix, binary outcome, and two pd-independent drivers (b1,b2).
    df = df.copy()
    if name == "telco":
        y = (df["Churn"].astype(str).str.strip() == "Yes").astype(int).values
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        X = df.drop(columns=["customerID", "Churn"]); b2col = "TotalCharges"
    elif name == "adult":
        y = df["HIGH_INCOME"].astype(int).values
        drop = ["income", "HIGH_INCOME", "AGE_YEARS", "SEX_STD", "EDU_STD", "VIP", "fnlwgt"]
        X = df.drop(columns=[c for c in drop if c in df]); b2col = "hours-per-week"
    elif name == "credit_risk":
        y = df["loan_status"].astype(int).values
        X = df.drop(columns=["loan_status"]); b2col = "person_income"
    elif name == "gmsc":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP"] if c in df]); b2col = "INCOME"
    elif name == "taiwan":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP", "AGE_YEARS", "SEX_STD"] if c in df]); b2col = "LIMIT_BAL"
    b2 = pd.to_numeric(df[b2col], errors="coerce").values if b2col in df else None
    Xn = pd.get_dummies(X, dummy_na=False).apply(pd.to_numeric, errors="coerce")
    return Xn, y, b2

def system_pd(X, y):
    # L2 logistic system model; out-of-fold PD via 5-fold stratified CV; in-fold median imputation.
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    pdh = np.zeros(len(y)); skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for tr, te in skf.split(X, y):
        Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
        med = Xtr.median(numeric_only=True); Xtr = Xtr.fillna(med); Xte = Xte.fillna(med)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
        clf.fit(sc.transform(Xtr), y[tr]); pdh[te] = clf.predict_proba(sc.transform(Xte))[:, 1]
    return pdh, roc_auc_score(y, pdh)

def pick_b1(X, pdh):
    # Near-orthogonal driver: numeric feature with the lowest |Spearman| vs PD (Attr43 analogue).
    best, br = None, 2.0
    for c in X.columns:
        v = X[c].values.astype(float); m = ~np.isnan(v)
        if m.sum() < 100 or np.nanstd(v) == 0: continue
        r = abs(spearmanr(v[m], pdh[m]).statistic)
        if not np.isnan(r) and r < br: br, best = r, v
    return best

QGRID = [0, 0.10, 0.20, 0.35, 0.55, 0.75, 0.90, 1.0]
def assign_grades(pdh):
    qs = np.quantile(pdh, QGRID); qs[0] -= 1e-9; qs[-1] += 1e-9
    return np.digitize(pdh, qs[1:-1])
SCHED = np.array([0.10, 0.15, 0.20, 0.35, 0.50, 0.70, 0.80])  # ord 0 best .. ord 6 worst
OVR = 0.35

def rank01(v):
    return pd.Series(v).rank(pct=True).fillna(0.5).values

def expected_gap(pi, y):
    # Exact expected upgrade-minus-none default-rate gap (percentage points).
    return (np.sum(pi * y) / np.sum(pi) - y.mean()) * 100.0

def sim_once(pi_firm, grade, G, rng):
    # One generative override draw; returns signed grade_diff (negative = upgrade) and masks.
    N = len(grade)
    overridden = rng.random(N) < OVR
    up = overridden & (rng.random(N) < pi_firm)
    down = overridden & ~up
    notch = np.where(rng.random(N) < 0.70, 1, 2)
    gd = np.zeros(N, int); gd[up] = -notch[up]; gd[down] = notch[down]
    final = np.clip(grade + gd, 0, G - 1)
    return final - grade, overridden, up

def save_fig(fig, name):
    # Save grayscale figure as PNG and PDF at 600 dpi; no caption; no path printed.
    fig.savefig(FIG / (name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(FIG / (name + ".pdf"), bbox_inches="tight")
    plt.close(fig)


# 03_gap_vs_gini_law.ipynb -- the design-induced selection law: gap vs Gini and Gini*Var(Y)
sysm = pd.read_csv(TAB / "system_model.csv")
rep = pd.read_csv(TAB / "replication_gap.csv")
naive = rep[rep.dgm == "naive"].set_index("dataset")
indep = rep[rep.dgm == "b1"].set_index("dataset")
rows = []
for name in DATASETS:
    s = sysm[sysm.dataset == name].iloc[0]; pbar = s["base_rate_pct"]/100
    rows.append({"dataset": name, "domain": DOMAIN[name], "Gini": s["Gini"], "pbar": pbar,
                 "naive": naive.loc[name, "gap_mean"], "nlo": naive.loc[name, "gap_lo"],
                 "nhi": naive.loc[name, "gap_hi"], "indep": indep.loc[name, "gap_mean"],
                 "GiniVar": s["Gini"]*pbar*(1-pbar)*100})
R = pd.DataFrame(rows)
R = pd.concat([R, pd.DataFrame([{"dataset": "polish", "domain": "corporate credit",
    "Gini": POLISH["gini"], "pbar": POLISH["pbar"], "naive": POLISH["naive_gap"],
    "nlo": 1.94, "nhi": 3.36, "indep": POLISH["indep_gap"],
    "GiniVar": POLISH["gini"]*POLISH["pbar"]*(1-POLISH["pbar"])*100}])], ignore_index=True)
x = R["GiniVar"].values; yv = R["naive"].values
slope = np.sum(x*yv)/np.sum(x*x); r2 = 1 - np.sum((yv-slope*x)**2)/np.sum((yv-yv.mean())**2)
R.to_csv(TAB / "law_points.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for _, r in R.iterrows():
    c, m = STYLE[r["domain"]]
    ax.errorbar(r["Gini"], r["naive"], yerr=[[r["naive"]-r["nlo"]], [r["nhi"]-r["naive"]]],
                fmt=m, color=c, ms=10 if m == "*" else 8, capsize=3)
    ax.annotate(r["dataset"], (r["Gini"], r["naive"]), textcoords="offset points", xytext=(6, 4), fontsize=8)
ax.scatter(R["Gini"], R["indep"], marker="x", color="0.6", s=40)
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("System Gini (= 2*AUC - 1)"); ax.set_ylabel("Upgrade - none gap (pp)")
ax = axes[1]
xs = np.linspace(0, x.max()*1.08, 50)
ax.plot(xs, slope*xs, color="0.1", lw=1.5, label="gap = %.2f * (Gini*Var),  R2=%.3f" % (slope, r2))
for _, r in R.iterrows():
    c, m = STYLE[r["domain"]]
    ax.errorbar(r["GiniVar"], r["naive"], yerr=[[r["naive"]-r["nlo"]], [r["nhi"]-r["naive"]]],
                fmt=m, color=c, ms=14 if m == "*" else 9, capsize=3)
    ax.annotate(r["dataset"], (r["GiniVar"], r["naive"]), textcoords="offset points", xytext=(6, -10), fontsize=8)
ax.scatter(R["GiniVar"], R["indep"], marker="x", color="0.6", s=40, label="pd-independent (b1)")
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("Gini * base rate * (1 - base rate)  (pp)"); ax.set_ylabel("Upgrade - none gap (pp)")
ax.legend(fontsize=8, loc="upper left")
save_fig(fig, "fig03_gap_vs_gini_law")
R


,dataset,domain,Gini,pbar,naive,nlo,nhi,indep,GiniVar
0,telco,non-credit,0.6887,0.2654,13.271,9.588,16.357,-0.097,13.427091
1,adult,non-credit,0.8119,0.2393,14.785,13.483,16.078,0.026,14.779463
2,credit_risk,consumer credit,0.7411,0.2182,12.798,11.319,14.228,-0.076,12.642333
3,gmsc,consumer credit,0.5806,0.0668,3.600,3.200,4.035,-0.361,3.619330
4,taiwan,consumer credit,0.4465,0.2212,7.805,6.219,9.203,0.066,7.691881
5,polish,corporate credit,0.5560,0.0482,2.720,1.940,3.360,0.330,2.550748
